# Stage 02 — features and targets

From `raw_close.csv`: log returns for the five ETFs, VIX/rate levels and
changes, yield-curve slope (10y − 3m), a credit-spread proxy (HYG/LQD), and
trailing realized volatility rv1/rv5/rv21 (HAR components, Corsi 2009).
Forward targets `y_rv{1,5,21}` use only days t+1..t+h, so features at t never
see the target window. Writes `data/processed/dataset.csv` (3873 × 21).


In [1]:
import os, pandas as pd, numpy as np

# Environment bootstrap: Colab (mount Drive) or a local checkout.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/volatility-forecast"
except ModuleNotFoundError:
    BASE = os.path.abspath(os.getcwd())          # find the repo root locally
    while not os.path.isdir(os.path.join(BASE, "src")):
        parent = os.path.dirname(BASE)
        if parent == BASE:
            raise FileNotFoundError("repo root with src/ not found")
        BASE = parent

RAW  = f"{BASE}/data/raw/raw_close.csv"
PROC = f"{BASE}/data/processed"
TICKERS = ["QQQ","^VIX","^TNX","^IRX","HYG","LQD","TLT","GLD"]

if os.path.exists(RAW):
    raw = pd.read_csv(RAW, index_col=0, parse_dates=True)
else:
    import yfinance as yf
    raw = yf.download(TICKERS, start="2011-01-01", end="2026-05-29",
                      auto_adjust=True)["Close"][TICKERS]

print(raw.shape)
print(raw.tail(2))

(3874, 8)
                   QQQ       ^VIX   ^TNX   ^IRX        HYG         LQD  \
Date                                                                     
2026-05-27  728.649292  16.290001  4.481  3.585  78.970665  107.672424   
2026-05-28  734.792480  15.740000  4.455  3.590  79.069221  107.998611   

                  TLT         GLD  
Date                               
2026-05-27  84.313515  408.489990  
2026-05-28  84.748413  412.769989  


In [2]:
qqq_days = raw["QQQ"].dropna().index   # target asset = reference calendar
df = raw.reindex(qqq_days)

print("missing on QQQ calendar (report, don't drop):")
for c in df.columns:
    m = df[c].isna()
    if m.any():
        print(f"  {c}: {int(m.sum())} -> {list(df.index[m].date)}")

df = df.ffill()   # level series: missing day = no new info -> carry prior
print("remaining NaN after ffill:", int(df.isna().sum().sum()))

missing on QQQ calendar (report, don't drop):
  ^TNX: 1 -> [datetime.date(2016, 11, 11)]
  ^IRX: 1 -> [datetime.date(2016, 11, 11)]
remaining NaN after ffill: 0


In [3]:
ret = lambda s: np.log(s / s.shift(1))
out = pd.DataFrame(index=df.index)

for t in ["QQQ","HYG","LQD","TLT","GLD"]:   # tradable ETF -> log return
    out[f"{t.lower()}_ret"] = ret(df[t])

out["vix_lvl"] = df["^VIX"]; out["vix_chg"] = df["^VIX"].diff()   # mean-reverting level + change
out["tnx_lvl"] = df["^TNX"]; out["tnx_chg"] = df["^TNX"].diff()
out["irx_lvl"] = df["^IRX"]; out["irx_chg"] = df["^IRX"].diff()

out["slope_lvl"]  = df["^TNX"] - df["^IRX"]    # yield-curve slope 10y-3m
out["slope_chg"]  = out["slope_lvl"].diff()
out["credit_lvl"] = df["HYG"] / df["LQD"]      # credit-spread proxy
out["credit_chg"] = out["credit_lvl"].diff()

In [4]:
sq = out["qqq_ret"] ** 2

for w in [1, 5, 21]:    # trailing RV, info up to t: HAR daily/weekly/monthly
    out[f"rv{w}"] = np.sqrt(sq.rolling(w).mean())

for h in [1, 5, 21]:    # forward target, info t+1..t+h: non-overlapping window len h
    out[f"y_rv{h}"] = np.sqrt(sq.rolling(h).sum().shift(-h) / h)

In [5]:
feat = [c for c in out.columns if not c.startswith("y_")]
tgt  = [c for c in out.columns if c.startswith("y_")]

print("head NaN (trailing-window warmup):"); print(out[feat].isna().sum())
print("\ntail NaN (forward horizon = live-prediction rows):"); print(out[tgt].isna().sum())
print("\nlast row w/ full features:", out[feat].dropna().index.max().date())
print("last 3 dates:", list(out.index[-3:].date))

os.makedirs(PROC, exist_ok=True)
out.to_csv(f"{PROC}/dataset.csv")
print("\nsaved:", out.shape, "->", f"{PROC}/dataset.csv")
print(out.columns.tolist())

head NaN (trailing-window warmup):
qqq_ret        1
hyg_ret        1
lqd_ret        1
tlt_ret        1
gld_ret        1
vix_lvl        0
vix_chg        1
tnx_lvl        0
tnx_chg        1
irx_lvl        0
irx_chg        1
slope_lvl      0
slope_chg      1
credit_lvl     0
credit_chg     1
rv1            1
rv5            5
rv21          21
dtype: int64

tail NaN (forward horizon = live-prediction rows):
y_rv1      1
y_rv5      5
y_rv21    21
dtype: int64

last row w/ full features: 2026-05-28
last 3 dates: [datetime.date(2026, 5, 26), datetime.date(2026, 5, 27), datetime.date(2026, 5, 28)]

saved: (3873, 21) -> /Users/jayden/Desktop/volatility-forecast/data/processed/dataset.csv
['qqq_ret', 'hyg_ret', 'lqd_ret', 'tlt_ret', 'gld_ret', 'vix_lvl', 'vix_chg', 'tnx_lvl', 'tnx_chg', 'irx_lvl', 'irx_chg', 'slope_lvl', 'slope_chg', 'credit_lvl', 'credit_chg', 'rv1', 'rv5', 'rv21', 'y_rv1', 'y_rv5', 'y_rv21']
